In [ ]:
"""
Sankey Plot: Design Principle → Descriptive Codes → Student Reactions
Data source: db-q4.csv
Requirements: pip install pandas plotly kaleido
"""

import pandas as pd
import plotly.graph_objects as go

int_var = "Exact question"
int_var2 = "Descriptive Codes"

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv("db-q4-2.csv")
df.columns = df.columns.str.strip()
df = df.dropna(subset=["Design Principle", int_var, "Students reactions"])

# ── 2. Build node list (order: Design Principles → Descriptive Codes → Reactions)
design_principles = sorted(df["Design Principle"].unique().tolist())
descriptive_codes = sorted(df[int_var].unique().tolist())
student_reactions = sorted(df["Students reactions"].unique().tolist())

all_nodes = design_principles + descriptive_codes + student_reactions

node_index = {name: i for i, name in enumerate(all_nodes)}

dp_count  = len(design_principles)
dc_count  = len(descriptive_codes)
sr_count  = len(student_reactions)

# ── 3. Build links ────────────────────────────────────────────────────────────
# Layer 1 → 2 : Design Principle → Descriptive Code
layer1 = (
    df.groupby(["Design Principle", int_var])
    .size()
    .reset_index(name="value")
)

# Layer 2 → 3 : Descriptive Code → Student Reaction
layer2 = (
    df.groupby([int_var, "Students reactions"])
    .size()
    .reset_index(name="value")
)

sources, targets, values = [], [], []

for _, row in layer1.iterrows():
    sources.append(node_index[row["Design Principle"]])
    targets.append(node_index[row[int_var]])
    values.append(row["value"])

for _, row in layer2.iterrows():
    sources.append(node_index[row[int_var]])
    targets.append(node_index[row["Students reactions"]])
    values.append(row["value"])

# ── 4. Colour palettes ────────────────────────────────────────────────────────
dp_colors = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"
]
dc_colors = ["#A8C8E8"] * dc_count          # soft blue for descriptive codes
sr_colors = [
    "#E07B54", "#6BAED6", "#74C476", "#9E9AC8",
    "#FD8D3C", "#FDAE6B", "#31A354", "#756BB1", "#636363"
][:sr_count]

node_colors = dp_colors + dc_colors + sr_colors

# Link colours: inherit source node colour with transparency
def hex_to_rgba(hex_color, alpha=0.35):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(node_colors[s]) for s in sources]

# ── 5. Build figure ───────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=22,
        line=dict(color="white", width=0.5),
        label=all_nodes,
        color=node_colors,
        hovertemplate="<b>%{label}</b><br>Total flow: %{value}<extra></extra>",
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        hovertemplate=(
            "<b>%{source.label}</b> → <b>%{target.label}</b>"
            "<br>Count: %{value}<extra></extra>"
        ),
    ),
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Design Principle → Descriptive Codes → Student Reactions</b>"
            "<br><sup>Each band width is proportional to the number of observations</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=16),
    ),
    font=dict(family="Arial", size=11, color="#333333"),
    paper_bgcolor="white",
    height=900,
    width=1400,
    margin=dict(l=20, r=20, t=90, b=20),
)

# ── 6. Add layer labels as annotations ───────────────────────────────────────
for x_pos, label in zip([0.01, 0.46, 0.99], ["Design Principles", int_var, "Student Reactions"]):
    fig.add_annotation(
        x=x_pos, y=1.04,
        xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=13, color="#444"),
        align="center",
    )

# ── 7. Save outputs ───────────────────────────────────────────────────────────
fig.write_html("sankey_output.html")
print("✅  Saved: sankey_output.html  (interactive)")

try:
    fig.write_image("sankey_output.png", scale=2)
    print("✅  Saved: sankey_output.png   (static, 2× resolution)")
except Exception as e:
    print(f"⚠️  PNG export skipped ({e}). Install kaleido: pip install kaleido")

fig.show()


✅  Saved: sankey_output.html  (interactive)
⚠️  PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Install kaleido: pip install kaleido


✅  Saved: sankey_output.html  (interactive)
⚠️  PNG export skipped (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Install kaleido: pip install kaleido
